# 2-1. 단일표본 t-검정과 독립표본 t-검정

- 단일표본 t-검정으로 성인 여성 키의 평균이 기준값 163cm와 다른지 확인한다.
- 독립표본 t-검정으로 A반과 B반의 평균 점수 차이를 확인한다.

In [14]:
import numpy as np          # 평균, 표준편차 등 수치 계산
import pandas as pd         # CSV 파일을 표 형태로 읽기
from scipy import stats     # t-검정, 정규성 검정 등 통계 함수

In [15]:
# 텍스트 파일을 읽기 모드("r")로 연다.
with open("datas2/성인여성_키_데이터.txt", "r") as file:
    # 파일 전체를 읽은 뒤, 줄바꿈 기준으로 나눈다.
    height_text = file.read().split("\n")

# 문자열로 읽힌 키 데이터를 실수(float) 리스트로 변환한다.
heights = list(map(float, height_text))

# 데이터 일부와 전체 개수를 확인한다.
print("앞에서 5개:", heights[:5])
print("데이터 개수:", len(heights))

앞에서 5개: [150.27, 142.94, 160.99, 157.48, 153.46]
데이터 개수: 25


## 1. 단일표본 t-검정

성인 여성 키 표본의 평균이 기준값 163cm와 통계적으로 다른지 확인한다.

- 귀무가설(H₀): 성인 여성 키의 평균은 163cm이다.
- 대립가설(H₁): 성인 여성 키의 평균은 163cm와 다르다.

In [16]:
# 표본 키 데이터의 평균을 계산한다.
height_mean = np.mean(heights)

# 표본 표준편차를 계산한다.
# ddof=1은 표본 데이터에서 사용하는 표준편차 설정이다.
height_std = np.std(heights, ddof=1)

print(f"평균 키: {height_mean:.2f}cm")
print(f"표본 표준편차: {height_std:.2f}cm")

평균 키: 156.93cm
표본 표준편차: 10.18cm


### 1-1. 정규성 검정

단일표본 t-검정은 표본 데이터가 정규분포에서 크게 벗어나지 않는다는 가정을 사용한다.

- 귀무가설(H₀): 키 데이터는 정규분포를 따른다.
- 대립가설(H₁): 키 데이터는 정규분포를 따르지 않는다.

In [17]:
# Shapiro-Wilk 정규성 검정을 수행한다.
# 결과는 검정통계량(statistic)과 p-value를 담고 있다.
normality_result = stats.shapiro(heights)

print(f"검정통계량: {normality_result.statistic:.4f}")
print(f"p-value: {normality_result.pvalue:.4f}")

# 유의수준 0.05를 기준으로 해석한다.
if normality_result.pvalue > 0.05:
    print("정규분포를 따르지 않는다는 충분한 증거가 없다.")
else:
    print("정규분포를 따르지 않는다고 판단한다.")

검정통계량: 0.9536
p-value: 0.3014
정규분포를 따르지 않는다는 충분한 증거가 없다.


### 1-2. 단일표본 t-검정 수행

유의수준 0.05에서 성인 여성 키의 평균이 163cm와 다른지 검정한다.

In [18]:
# 기준이 되는 모집단 평균을 설정한다.
reference_mean = 163

# 단일표본 t-검정을 수행한다.
# heights의 평균이 reference_mean과 다른지 양측 검정한다.
one_sample_result = stats.ttest_1samp(heights, popmean=reference_mean)

print(f"t-통계량: {one_sample_result.statistic:.4f}")
print(f"p-value: {one_sample_result.pvalue:.4f}")

# 유의수준 0.05를 기준으로 가설검정 결과를 해석한다.
if one_sample_result.pvalue <= 0.05:
    print("귀무가설을 기각한다: 평균 키는 163cm와 통계적으로 다르다.")
else:
    print("귀무가설을 기각하지 못한다: 평균 키가 163cm와 다르다고 할 충분한 증거가 없다.")

t-통계량: -2.9798
p-value: 0.0065
귀무가설을 기각한다: 평균 키는 163cm와 통계적으로 다르다.


## 2. 독립표본 t-검정

A반과 B반은 서로 다른 학생으로 구성된 독립된 두 그룹이다.
두 반의 평균 점수가 통계적으로 다른지 확인한다.

- 귀무가설(H₀): A반과 B반의 평균 점수는 같다.
- 대립가설(H₁): A반과 B반의 평균 점수는 다르다.

In [19]:
# 반별 점수 CSV 파일을 DataFrame 형태로 읽는다.
# 한글 인코딩 문제를 피하기 위해 euc-kr 인코딩을 지정한다.
scores_df = pd.read_csv("datas2/반별_점수_type1.csv", encoding="euc-kr")

# 데이터의 앞부분을 확인한다.
scores_df.head()

,반,점수
0,A,73
1,A,69
2,A,71
3,A,71
4,A,73


In [20]:
# '반' 열의 값이 A인 행만 골라 점수 배열을 만든다.
group_a = scores_df.loc[scores_df["반"] == "A", "점수"].to_numpy()

# '반' 열의 값이 B인 행만 골라 점수 배열을 만든다.
group_b = scores_df.loc[scores_df["반"] == "B", "점수"].to_numpy()

# 각 그룹의 데이터 개수와 일부 점수를 확인한다.
print("A반 인원:", len(group_a))
print("A반 점수 일부:", group_a[:5])

print("B반 인원:", len(group_b))
print("B반 점수 일부:", group_b[:5])

A반 인원: 20
A반 점수 일부: [73 69 71 71 73]
B반 인원: 10
B반 점수 일부: [63 56 73 61 55]


In [21]:
# 각 반의 평균 점수를 계산한다.
mean_a = np.mean(group_a)
mean_b = np.mean(group_b)

# 각 반의 표본 표준편차를 계산한다.
std_a = np.std(group_a, ddof=1)
std_b = np.std(group_b, ddof=1)

print(f"A반 평균: {mean_a:.2f}, 표준편차: {std_a:.2f}")
print(f"B반 평균: {mean_b:.2f}, 표준편차: {std_b:.2f}")
print(f"평균 차이(A반 - B반): {mean_a - mean_b:.2f}")

A반 평균: 70.55, 표준편차: 5.68
B반 평균: 64.10, 표준편차: 8.28
평균 차이(A반 - B반): 6.45


### 2-1. 정규성 검정

독립표본 t-검정 전 각 그룹의 점수 분포가 정규분포에서 크게 벗어나지 않는지 확인한다.

- 귀무가설(H₀): 해당 그룹의 점수는 정규분포를 따른다.
- 대립가설(H₁): 해당 그룹의 점수는 정규분포를 따르지 않는다.

In [22]:
# A반과 B반 각각에 Shapiro-Wilk 정규성 검정을 수행한다.
normality_a = stats.shapiro(group_a)
normality_b = stats.shapiro(group_b)

print(f"A반 p-value: {normality_a.pvalue:.4f}")
print(f"B반 p-value: {normality_b.pvalue:.4f}")

# 두 그룹의 정규성 검정 결과를 함께 해석한다.
if normality_a.pvalue > 0.05 and normality_b.pvalue > 0.05:
    print("두 그룹 모두 정규성을 위반한다고 볼 충분한 증거가 없다.")
else:
    print("적어도 한 그룹은 정규성을 위반할 가능성이 있다.")

A반 p-value: 0.7485
B반 p-value: 0.1646
두 그룹 모두 정규성을 위반한다고 볼 충분한 증거가 없다.


### 2-2. 등분산성 검정

독립표본 t-검정에서 두 그룹의 분산이 같은지 확인한다.

- 귀무가설(H₀): A반과 B반의 점수 분산은 같다.
- 대립가설(H₁): A반과 B반의 점수 분산은 다르다.

In [23]:
# Levene 검정으로 두 그룹의 등분산성을 확인한다.
variance_result = stats.levene(group_a, group_b)

print(f"검정통계량: {variance_result.statistic:.4f}")
print(f"p-value: {variance_result.pvalue:.4f}")

# p-value를 기준으로 t-검정의 equal_var 설정을 결정한다.
if variance_result.pvalue > 0.05:
    equal_variance = True
    print("등분산성을 위반한다고 볼 충분한 증거가 없다.")
else:
    equal_variance = False
    print("두 그룹의 분산이 다르다고 판단한다. Welch t-검정을 사용한다.")

검정통계량: 2.0331
p-value: 0.1650
등분산성을 위반한다고 볼 충분한 증거가 없다.


### 2-3. 독립표본 t-검정 수행

유의수준 0.05에서 A반과 B반의 평균 점수가 다른지 양측 검정한다.

In [24]:
# A반과 B반의 평균 점수 차이에 대해 독립표본 t-검정을 수행한다.
# equal_variance 값은 바로 앞 Levene 검정 결과를 반영한다.
independent_result = stats.ttest_ind(
    group_a,
    group_b,
    equal_var=equal_variance
)

print(f"t-통계량: {independent_result.statistic:.4f}")
print(f"p-value: {independent_result.pvalue:.4f}")

# 유의수준 0.05를 기준으로 검정 결과를 해석한다.
if independent_result.pvalue <= 0.05:
    print("귀무가설을 기각한다: A반과 B반의 평균 점수는 통계적으로 다르다.")
else:
    print("귀무가설을 기각하지 못한다: 평균 점수가 다르다고 할 충분한 증거가 없다.")

t-통계량: 2.5129
p-value: 0.0180
귀무가설을 기각한다: A반과 B반의 평균 점수는 통계적으로 다르다.


### 2-4. 독립표본 t-검정 결과

- A반 평균은 70.55점, B반 평균은 64.10점이었다.
- 두 반의 평균 차이는 6.45점이었다.
- p-value가 0.0180으로 유의수준 0.05보다 작았다.
- 따라서 귀무가설을 기각했다.
- A반과 B반의 평균 점수는 통계적으로 다르며, A반의 평균 점수가 더 높았다.

In [25]:
# A반과 B반 점수가 각각 별도 열에 있는 CSV 파일을 읽는다.
scores_wide_df = pd.read_csv(
    "datas2/반별_점수_type2.csv",
    encoding="euc-kr"
)

# 데이터 앞부분을 확인한다.
scores_wide_df.head()

,A반,B반
0,73,63.0
1,69,56.0
2,71,73.0
3,71,61.0
4,73,55.0


In [26]:
# A반 열에서 빈칸을 제거한 뒤 NumPy 배열로 변환한다.
group_a_wide = scores_wide_df["A반"].dropna().to_numpy()

# B반 열의 빈칸도 제거한 뒤 NumPy 배열로 변환한다.
group_b_wide = scores_wide_df["B반"].dropna().to_numpy()

print("A반 인원:", len(group_a_wide))
print("B반 인원:", len(group_b_wide))

# 앞에서 type1 파일로 만든 그룹과 같은 데이터인지 확인한다.
print("A반 데이터 동일 여부:", np.array_equal(group_a, group_a_wide))
print("B반 데이터 동일 여부:", np.array_equal(group_b, group_b_wide))

A반 인원: 20
B반 인원: 10
A반 데이터 동일 여부: True
B반 데이터 동일 여부: True
